# 15 · Crop lifecycle and renewal opportunity
**Open feature research · one bounded development block · no automatic promotion.**

Notebook 14 changed no sampled actions, so its pilot was skipped. This milestone separately tests remaining productive lifetime, removal of wasted maintenance, and earlier clearing of empty exhausted plants. It does not repeat the broad WATER-deferral rule.

Run with **Kaggriculture Manual (verified source)**. No installs, cloud changes, downloads or prior-study reruns. See START_HERE.md and FEATURE_RESEARCH.md.

In [1]:
from pathlib import Path
import json, os, signal, subprocess, sys, time
import pandas as pd
import plotly.io as pio
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / 'run_lifecycle.py').exists():
    ROOT = Path.home() / 'kaggriculture_crop_lifecycle'
if not (ROOT / 'run_lifecycle.py').exists():
    raise FileNotFoundError('Open the notebook from the extracted package folder.')
sys.path.insert(0, str(ROOT))
import visualize
if Path(visualize.__file__).resolve() != ROOT / 'visualize.py':
    raise RuntimeError('Another notebook module is loaded. Restart this kernel, then run again.')
from visualize import reference_figures, screen_figures, pilot_figures, dashboard
OUT = ROOT / 'outputs'
pio.renderers.default = 'plotly_mimetype'
figures = []
receipt = Path.home() / 'kaggriculture_manual_resume/state/runtime.json'
PYTHON_BIN = json.loads(receipt.read_text())['executable']
if Path(sys.executable).absolute() != Path(PYTHON_BIN).absolute():
    raise RuntimeError('Select Kaggriculture Manual (verified source), then restart the kernel.')
print('LIVE MANUAL WORKSPACE | existing data; no prior-study reruns')
print('Source folder:', ROOT)
print('Interpreter:', PYTHON_BIN)

LIVE MANUAL WORKSPACE | existing data; no prior-study reruns
Source folder: /home/sagemaker-user/kaggriculture_crop_lifecycle
Interpreter: /home/sagemaker-user/projects/kaggriculture/.venv/bin/python


## 1 · Prior evidence: nonactivation is not an endpoint experiment

In [2]:
previous = json.loads((ROOT / 'reference/notebook14_screen_report.json').read_text())
display(pd.DataFrame([{k: previous[k] for k in ['status','decision','observations','task_rows','action_changes','candidate_callback_max_ms','elapsed_seconds']}]))
print('The previous pilot completed zero pairs. No new leaderboard score is recorded.')

,status,decision,observations,task_rows,action_changes,candidate_callback_max_ms,elapsed_seconds
0,HARVEST_VALUE_SCREEN_COMPLETE,STOP_NO_ACTION_ACTIVATION,156,287,0,33.085842,11.529806


The previous pilot completed zero pairs. No new leaderboard score is recorded.


In [3]:
for figure in reference_figures(ROOT):
    figure.show()
    figures.append(figure)

## 2 · Controlled component comparisons
**Control:** ordinary behavior plus the final-sale correction. **Retire:** remove WATER jobs only on empty ongoing plants with no lifetime production left. **Renew:** retire plus existing crop-layout DIG eligibility on those plants.

Held goods and future producers are protected. Crop mix, priority constants, allocator, hiring and market rules stay fixed. Treatment days 8–27. Nominal events assume survival; the logged earliest-sale step is a lower bound, not a forecast.

In [4]:
display(pd.Series(json.loads((ROOT / 'PROTOCOL.json').read_text()), name='registered design').to_frame())
display(pd.read_csv(ROOT / 'feature_dictionary.csv')[['feature','scope','family','role']])

,registered design
milestone,15
name,crop-lifecycle
source_commit,7194116dfc92a8663139611233b5a221dad431a4
engine_sha256,bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d...
active_days,"[8, 27]"
screen_stride_callbacks,4
screen_boundaries,"[191, 672, 695, 696]"
screen_sources,"[[1601, 0], [1601, 1], [1602, 0]]"
select,first activated key by seed/seat; no rewards used
arms,"[control, retire, renew]"


,feature,scope,family,role
0,ongoing,plant,productive_lifetime,candidate_context_or_diagnostic
1,age_days,plant,productive_lifetime,candidate_context_or_diagnostic
2,held_units,plant,productive_lifetime,candidate_context_or_diagnostic
3,harvestable_units,plant,productive_lifetime,candidate_context_or_diagnostic
4,watered_today,plant,productive_lifetime,candidate_context_or_diagnostic
5,dry_streak,plant,productive_lifetime,candidate_context_or_diagnostic
6,fertilizer_active,plant,productive_lifetime,candidate_context_or_diagnostic
7,first_production_day,plant,productive_lifetime,candidate_context_or_diagnostic
8,last_production_day,plant,productive_lifetime,candidate_context_or_diagnostic
9,lifetime_event_count,plant,productive_lifetime,candidate_context_or_diagnostic


## 3 · Bounded stage helper: live heartbeats and saved diagnostics

In [5]:
# The runner owns hard caps; file streaming keeps notebook output responsive.
def run_stage(stage):
    caps = {'screen': 120, 'pilot': 180}
    if stage not in caps:
        raise ValueError('Unregistered stage')
    OUT.mkdir(exist_ok=True)
    logpath = OUT / (stage + '_notebook_console.txt')
    process = None
    offset = 0
    started = time.monotonic()
    def show_new():
        nonlocal offset
        with logpath.open('r', errors='replace') as stream:
            stream.seek(offset)
            text = stream.read()
            offset = stream.tell()
        if text:
            print(text, end='', flush=True)
    try:
        with logpath.open('w') as console:
            process = subprocess.Popen(
                [PYTHON_BIN, '-u', str(ROOT / 'run_lifecycle.py'), stage],
                cwd=ROOT, stdout=console, stderr=subprocess.STDOUT,
                start_new_session=True,
            )
            while process.poll() is None:
                show_new()
                if time.monotonic() - started > caps[stage] + 30:
                    raise TimeoutError('Notebook emergency deadline; save and bundle diagnostics.')
                time.sleep(0.25)
            show_new()
            if process.returncode:
                raise RuntimeError(f'{stage} failed. Stop, save and bundle. See {logpath}.')
    except BaseException:
        if process is not None and process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise
    return json.loads((OUT / stage / 'report.json').read_text())

## 4 · Mechanics and activation screen
380 calendar fixtures and four DIG/replant fixtures must pass against installed engine components. Screen 372 fixed sparse observations. Control/null/source actions must agree, with changes confined to the registered window.

In [6]:
screen = run_stage('screen')
display(pd.DataFrame([{k: screen[k] for k in ['status','decision','observations','source_episodes','action_changes','candidate_callback_max_ms','elapsed_seconds']}]))
mechanics = json.loads((OUT / 'mechanics.json').read_text())
print('Installed-engine component fixtures:', mechanics['calendar_cases'], '+', mechanics['dig_replant_cases'])

....................................................
----------------------------------------------------------------------
Ran 52 tests in 7.685s

OK
{"utc": "2026-09-12T21:57:38.718872+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10}
{"utc": "2026-09-12T21:57:38.762490+00:00", "stage": "MECHANICS_PASSED", "calendar_cases": 380, "dig_replant_cases": 4}
{"utc": "2026-09-12T21:57:39.441097+00:00", "stage": "HEARTBEAT", "name": "screen", "elapsed_seconds": 10.0}
{"utc": "2026-09-12T21:57:43.233150+00:00", "stage": "SCREEN_EPISODE_SAVED", "seed": 1601, "seat": 0, "opponent": "livestock_fertilizer", "arm": "coordinated", "states": 124}
{"utc": "2026-09-12T21:57:46.888819+00:00", "stage": "SCREEN_EPISODE_SAVED", "seed": 1601, "seat": 1, "opponent": "livestock_fertilizer", "arm": "coordinated", "states": 124}
{"utc": "2026-09-12T21:57:49.445648+00:00", "stage": "HEARTBEAT", "name": "screen", "elapsed_seconds": 20.0}
{"utc": "2026-09-12T21:57:51.003414+00:00", "stage": "SCREEN_EPI

,status,decision,observations,source_episodes,action_changes,candidate_callback_max_ms,elapsed_seconds
0,CROP_LIFECYCLE_SCREEN_COMPLETE,ELIGIBLE_FOR_ONE_THREE_ARM_BLOCK,372,3,7,86.505571,21.676997


Installed-engine component fixtures: 380 + 4


In [7]:
for figure in screen_figures(ROOT):
    figure.show()
    figures.append(figure)

## 5 · One three-arm block, only after activation
Choose the first activated seed/seat without rewards. Replay 192 prefix transitions, then advance 527 responsive decisions for each of control, retire and renew. Completed branches receive checksummed checkpoints. These are correlated development contrasts, not untouched validation.

A harmful arm is rejected, not retried or scaled. Complete only this prespecified block so the component comparison remains available. No automatic promotion.

In [8]:
pilot = run_stage('pilot')
print('Block decision:', pilot['decision'])
print('Completed branches:', pilot['completed_branches'])
if pilot['completed_branches']:
    results = pd.read_csv(OUT / 'pilot/paired_results.csv')
    display(results[['contrast','coins_control','coins_candidate','coins_delta','coin_margin_delta','local_match_score_delta','decision']])
    display(pd.Series(pilot['renew_minus_retire'], name='renew minus retire: component effect').to_frame())
else:
    print('No action activation: all continuations skipped; no endpoint benefit measured.')

{"utc": "2026-09-12T21:57:52.748154+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10}
{"utc": "2026-09-12T21:57:54.082993+00:00", "stage": "PREFIX_REPLAY", "mode": "control", "completed": 120, "total": 192}
{"utc": "2026-09-12T21:57:56.565924+00:00", "stage": "SUFFIX_PROGRESS", "mode": "control", "completed": 48, "total": 527}
{"utc": "2026-09-12T21:57:58.564091+00:00", "stage": "SUFFIX_PROGRESS", "mode": "control", "completed": 96, "total": 527}
{"utc": "2026-09-12T21:58:00.499607+00:00", "stage": "SUFFIX_PROGRESS", "mode": "control", "completed": 144, "total": 527}
{"utc": "2026-09-12T21:58:01.977505+00:00", "stage": "HEARTBEAT", "name": "pilot", "elapsed_seconds": 10.0}
{"utc": "2026-09-12T21:58:02.249767+00:00", "stage": "SUFFIX_PROGRESS", "mode": "control", "completed": 192, "total": 527}
{"utc": "2026-09-12T21:58:03.972652+00:00", "stage": "SUFFIX_PROGRESS", "mode": "control", "completed": 240, "total": 527}
{"utc": "2026-09-12T21:58:05.813846+00:00", "stage": "SUFFIX_

,contrast,coins_control,coins_candidate,coins_delta,coin_margin_delta,local_match_score_delta,decision
0,retire-control,44840.0,41510.0,-3330.0,-488.0,0.0,STOP_NEGATIVE_ENDPOINT
1,renew-control,44840.0,43451.0,-1389.0,1054.0,0.0,STOP_NEGATIVE_ENDPOINT


,renew minus retire: component effect
coin_margin,1542.0
coins,1941.0
local_match_score,0.0
opponent_coins,399.0
residual_product_units,0.0


In [9]:
for figure in pilot_figures(ROOT):
    figure.show()
    figures.append(figure)

## 6 · Save, bundle, stop compute
No result here is an official Kaggle rating. Fresh groups, different opponents, whole-agent workforce coverage and a verified submission remain open. Save with Ctrl+S before bundling. Stop the AWS application, not the space.

In [10]:
path = dashboard(figures, OUT / 'crop_lifecycle_dashboard.html')
review = {'screen_decision': screen['decision'], 'pilot_decision': pilot['decision'], 'plots': len(figures),
          'official_submission_score': None, 'github_updated': False, 'feature_engineering_complete': False,
          'next_action': 'Save notebook, bundle results, then stop the AWS application.'}
(OUT / 'notebook_review.json').write_text(json.dumps(review, indent=2))
print('Dashboard:', path)
print('NOTEBOOK15_COMPLETE')
print(json.dumps(review, indent=2))

Dashboard: /home/sagemaker-user/kaggriculture_crop_lifecycle/outputs/crop_lifecycle_dashboard.html
NOTEBOOK15_COMPLETE
{
  "screen_decision": "ELIGIBLE_FOR_ONE_THREE_ARM_BLOCK",
  "pilot_decision": "THREE_ARM_DEVELOPMENT_BLOCK_COMPLETE_NO_AUTOMATIC_PROMOTION",
  "plots": 10,
  "official_submission_score": null,
  "github_updated": false,
  "feature_engineering_complete": false,
  "next_action": "Save notebook, bundle results, then stop the AWS application."
}
